# Introduction

This notebook presents an implementaiton of the **indexing pipeline for a GraphRAG via Lexical Search**, using:
- Tech Stack: Neo4j and LangChain
- Query Method: Lexical Search via Cypher query language (not vector search/semantic search)

**Setting up**

You will need to have a running Neo4j instance. One option is to create a [free Neo4j database instance in their Aura cloud service](https://neo4j.com/cloud/platform/aura-graph-database/). You can also run the database locally using the [Neo4j Desktop application](https://neo4j.com/download/), or running a docker container. Check the references for more information.

**Sources**
1. [LangChain | How to construct knowledge graphs](https://python.langchain.com/v0.2/docs/how_to/graph_constructing/)
2. [Github | Neo4j GraphRAG Package for Python](https://github.com/neo4j/neo4j-graphrag-python)
3. [DeepLearning.AI | Knowledge Graphs for RAG](https://learn.deeplearning.ai/courses/knowledge-graphs-rag/lesson/5/constructing-a-knowledge-graph-from-text-documents)
4. [GraphRAG | Lexical Graph](https://graphr.ag/reference/knowledge-graph/lexical-graph/)

In [ ]:
! pip install --quiet \
              langchain==0.2.14 \
              langchain-openai==0.1.20 \
              langchain-community==0.2.12 \
              langchain-experimental==0.0.64 \
              neo4j-graphrag==1.2.0 \
              pymupdf==1.24.13 \
              pypdf==5.1.0 \
              python-dotenv==1.0.1

In [ ]:
import asyncio
import os
import requests

from dotenv import load_dotenv, find_dotenv
from langchain.chains import GraphCypherQAChain
from langchain.prompts.prompt import PromptTemplate
from langchain_community.graphs import Neo4jGraph
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_openai import ChatOpenAI
from neo4j import GraphDatabase
from neo4j_graphrag.embeddings import OpenAIEmbeddings
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline
from neo4j_graphrag.llm.openai_llm import OpenAILLM

In [ ]:
use_google_drive_api = True
google_drive_path = 'gdrive/My Drive/Colab Notebooks/Toolkit-NLP/GraphRAG_with_Neo4j_and_LangChain'

if(use_google_drive_api):
  from google.colab import drive
  drive.mount('/content/gdrive/', force_remount=True)
  # Changing the current working directory to the notebook folder
  os.chdir(google_drive_path)
  print(os.getcwd())

Mounted at /content/gdrive/
/content/gdrive/My Drive/Colab Notebooks/Toolkit-NLP/GraphRAG_with_Neo4j_and_LangChain


In [ ]:
# OpenAI API Key
# https://openai.com/pricing

_ = load_dotenv(find_dotenv(), override=True) # read local .env file
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
NEO4J_URI = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]

# 0. Create the dataset

In [ ]:
# List of URLs to download
pdf_urls = [
    "https://www.un.org/sustainabledevelopment/wp-content/uploads/2018/09/Goal-1.pdf",
    "https://www.un.org/sustainabledevelopment/wp-content/uploads/2018/09/Goal-2.pdf",
    "https://www.un.org/sustainabledevelopment/wp-content/uploads/2018/09/Goal-3.pdf",
    "https://www.un.org/sustainabledevelopment/wp-content/uploads/2018/09/Goal-4.pdf"
]

# Directory to save PDFs
data_dir = "data"

In [ ]:
# Create the directory to save PDFs if it doesn't exist
os.makedirs(data_dir, exist_ok=True)  #

# Headers to mimic a browser request
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/87.0.4280.88 Safari/537.36"
}

# Download each PDF
for url in pdf_urls:
    # Extract file name from URL
    file_name = os.path.join(data_dir, url.split("/")[-1])

    try:
        # Download the file and save it
        response = requests.get(url, headers=headers, timeout=10)  # Set a timeout for reliability
        response.raise_for_status()  # Check if the request was successful
        with open(file_name, "wb") as file:
            file.write(response.content)
        print(f"Downloaded: {file_name}")
    except requests.exceptions.RequestException as e:
        print(f"Failed to download {url}. Error: {e}")

Downloaded: data/Goal-1.pdf
Downloaded: data/Goal-2.pdf
Downloaded: data/Goal-3.pdf
Downloaded: data/Goal-4.pdf


# 1. Load and Split the PDF files

In [ ]:
import os
from langchain.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Initialize the text splitter with a max chunk size of 1000 characters and overlap of 200 characters
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# Function to load and split PDFs
def load_and_split_pdfs(directory):
    documents = []

    for file_name in os.listdir(directory):
        if file_name.endswith(".pdf"):
            # Load the PDF
            file_path = os.path.join(directory, file_name)
            loader = PyMuPDFLoader(file_path)
            pdf_docs = loader.load()

            # Split the loaded text into chunks
            for pdf_doc in pdf_docs:
                split_docs = text_splitter.split_documents([pdf_doc])
                documents.extend(split_docs)

    return documents

# Load and split the documents
chunks = load_and_split_pdfs(data_dir)

In [ ]:
len(chunks)

19

In [ ]:
chunks[0].page_content

'What’s the goal here?\nTo end poverty in all its \nforms everywhere by 2030.\nWhy?\nMore than 700 million peo\xad\nple, or 11% of the world pop\xad\nulation, still live in extreme \npoverty and is struggling to \nfulfil the most basic needs \nlike health, education, and \naccess to water and san\xad\nitation, to name a few.. \nYes. The overwhelming \nmajority of people living \non less than $1.90 a day \nlive in Southern Asia and \nsub-Saharan Africa and \nthey account for about 70 \nper cent of the global total \nof extremely poor people.\nIn 2017, economic losses \nattributed to natu\xad\nral disasters were esti\xad\nmated at over $300 \nbillion, among the highest \nlosses in recent years.\nHowever, this issue also \naffects developed coun\xad\ntries. Right now there \nare 30 million children \ngrowing up poor in the \nworld’s richest countries.\nBy 2030  \n167 million  \nchildren \nwill live in \nextreme \npoverty\nif the world\ndoesn’t take\n action \nto improve \nhealth \nand edu

In [ ]:
chunks[0].metadata

{'source': 'data/Goal-1.pdf',
 'file_path': 'data/Goal-1.pdf',
 'page': 0,
 'total_pages': 2,
 'format': 'PDF 1.4',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'creator': 'Adobe InDesign CC 13.1 (Macintosh)',
 'producer': 'Adobe PDF Library 15.0',
 'creationDate': "D:20180914103736-04'00'",
 'modDate': "D:20180914103737-04'00'",
 'trapped': ''}

# 2. Knowledge Graph Construction

## 2.1. Automatic

Notes:
- Usign LangChain integration.
- Embeddings are not generated.

In [ ]:
kg = Neo4jGraph(url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD)

In [ ]:
# Instantiate an LLM

llm = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    temperature=0.4,
    model_name="gpt-4o",
)

In [ ]:
# The LLMGraphTransformer converts text documents into structured graph documents
# by leveraging a LLM to parse and categorize entities and their relationships.

llm_transformer = LLMGraphTransformer(
  llm=llm,
  node_properties=True,
  relationship_properties=True
)

In [ ]:
graph_documents = llm_transformer.convert_to_graph_documents(chunks)

In [ ]:
print(f"Nodes:{graph_documents[0].nodes}")
print(f"Relationships:{graph_documents[0].relationships}")

Nodes:[Node(id='Inclusive_And_Quality_Education', type='Concept', properties={'goal': 'Ensure inclusive and quality education for all and promote lifelong learning'}), Node(id='Education', type='Concept', properties={'importance': 'Key to achieving Sustainable Development Goals (SDGs), breaking the cycle of poverty, reducing inequalities, reaching gender equality, empowering people to live healthy and sustainable lives, fostering tolerance, contributing to peaceful societies, enabling better jobs and lives'}), Node(id='Primary_Education', type='Concept', properties={'enrolment_rate': '91% in developing countries'}), Node(id='Children_Out_Of_School', type='Concept', properties={'number': '57 million'})]
Relationships:[Relationship(source=Node(id='Education', type='Concept'), target=Node(id='Inclusive_And_Quality_Education', type='Concept'), type='PROMOTES'), Relationship(source=Node(id='Education', type='Concept'), target=Node(id='Primary_Education', type='Concept'), type='IMPROVES'), R

In [ ]:
for gd in graph_documents:
  for relationship in gd.relationships:
    if relationship.source.type=='Amount' or relationship.target.type=='Amount':
      print(relationship)

source=Node(id='$175 Billion', type='Amount') target=Node(id='Richest Countries', type='Countries') type='INCOME_PERCENTAGE' properties={'percentage': 'less than one percent'}


In [ ]:
nodes = set()
relationships = set()
node_properties = set()
relationships_properties = set()
for gd in graph_documents:
  for node in gd.nodes:
    nodes.add(node.type)
    try:
      node_properties = node_properties.union(set(node.properties.keys()))
    except:
      pass

  for relationship in gd.relationships:
    relationships.add(relationship.type)
    try:
      relationships_properties = relationships_properties.union(set(relationship.properties.keys()))
    except:
      pass

nodes = list(nodes)
nodes.sort()

relationships = list(relationships)
relationships.sort()

node_properties = list(node_properties)
node_properties.sort()

relationships_properties = list(relationships_properties)
relationships_properties.sort()

print('Nodes:', nodes)
print('Nodes properties:', node_properties)
print('')
print('Relationships:', relationships)
print('Relationships properties:', relationships_properties)

Nodes: ['Amount', 'Barrier', 'Cause', 'Community', 'Concept', 'Condition', 'Countries', 'Disease', 'Economic loss', 'Education', 'Entity', 'Event', 'Future population', 'Goal', 'Group', 'Income', 'Initiative', 'Issue', 'Needs', 'Opportunity', 'Organization', 'Period', 'Person', 'Population', 'Projection', 'Quantity', 'Region', 'Resource', 'Role', 'Sector', 'Source', 'State', 'Statistic', 'System', 'Time', 'Time period', 'Website']
Nodes properties: ['age_range', 'description', 'enrolment_rate', 'goal', 'importance', 'number', 'percentage', 'target_year', 'year']

Relationships: ['ACCESS', 'ACCESS_13.6_MILLION_PEOPLE', 'ACCOUNTABLE_TO', 'ACHIEVE_BY', 'ACHIEVING', 'ADDRESS', 'AFFECTED_REGION', 'AVERTED_15.6_MILLION_DEATHS', 'BARRIERS_FOR_GIRLS', 'BARRIER_TO', 'CALCULATED_COST_TO_END', 'CAN_END', 'CAUSE', 'CAUSES', 'CHALLENGE', 'COMMITMENT', 'COMMIT_TO_PROVIDE_FREE', 'CONTRIBUTES_TO', 'CONTRIBUTION', 'CONTRIBUTOR', 'COST_TO_ACHIEVE', 'CRITICAL_FOR', 'DATA_SOURCE', 'DECLINED', 'DEMAND', 'D

## 2.2. Semi-Automatic
-> Fixed nodes and relationships

Notes:
- Usign LangChain integration.
- Embeddings are not generated.

In [ ]:
# The LLMGraphTransformer converts text documents into structured graph documents
# by leveraging a LLM to parse and categorize entities and their relationships.

allowed_nodes=["Chunk", "File", "Topic", "Person", "Organization", "Adress", 'Amount']
allowed_relationships=["NEXT", "PART_OF", "NATIONALITY", "LOCATED_AT", "WORKED_AT", "INCOME"]
node_properties=['content', 'description', 'source', 'page', 'year', "name", "street", "city", "state", "country", "region", "value"]
relationship_properties=['description', 'source', 'year']

llm_transformer = LLMGraphTransformer(
  llm=llm,
  allowed_nodes=allowed_nodes,
  allowed_relationships=allowed_relationships,
  node_properties=node_properties,
  relationship_properties=relationship_properties,
  strict_mode=True
)

In [ ]:
llm_transformer.Extract

In [ ]:
graph_documents = llm_transformer.convert_to_graph_documents(chunks)

In [ ]:
for node in graph_documents:
  print(node)

nodes=[Node(id='Endpovertygoal', type='Topic', properties={'description': 'To end poverty in all its forms everywhere by 2030.'}), Node(id='Extremepovertystats', type='Chunk', properties={'content': 'More than 700 million people, or 11% of the world population, still live in extreme poverty.'}), Node(id='Economiclosses2017', type='Amount', properties={'value': 'over $300 billion'}), Node(id='Developedcountriespoverty', type='Chunk', properties={'content': '30 million children growing up poor in the world’s richest countries.'}), Node(id='Futurepovertyprediction', type='Chunk', properties={'content': 'By 2030, 167 million children will live in extreme poverty if no action is taken.'}), Node(id='Nopoverty', type='Topic', properties={'description': 'NO POVERTY: WHY IT MATTERS'})] relationships=[Relationship(source=Node(id='Economiclosses2017', type='Amount'), target=Node(id='Extremepovertystats', type='Chunk'), type='NEXT'), Relationship(source=Node(id='Developedcountriespoverty', type='C

In [ ]:
kg.add_graph_documents(graph_documents)

kg.refresh_schema()

In [ ]:
print(kg.schema)

Node properties:
Topic {id: STRING, description: STRING, content: STRING, source: STRING}
Chunk {id: STRING, content: STRING}
Amount {id: STRING, value: STRING, description: STRING}
Person {id: STRING, name: STRING, description: STRING}
Organization {id: STRING, description: STRING, name: STRING}
File {id: STRING, source: STRING}
Relationship properties:
PART_OF {description: STRING, source: STRING}
The relationships:
(:Topic)-[:NEXT]->(:Topic)
(:Topic)-[:PART_OF]->(:Topic)
(:Topic)-[:PART_OF]->(:File)
(:Topic)-[:PART_OF]->(:Amount)
(:Topic)-[:PART_OF]->(:Chunk)
(:Topic)-[:INCOME]->(:Amount)
(:Chunk)-[:NEXT]->(:Chunk)
(:Chunk)-[:NEXT]->(:Topic)
(:Chunk)-[:PART_OF]->(:Topic)
(:Amount)-[:NEXT]->(:Chunk)
(:Amount)-[:PART_OF]->(:Topic)
(:Amount)-[:PART_OF]->(:Person)
(:Amount)-[:PART_OF]->(:Amount)
(:Amount)-[:INCOME]->(:Organization)
(:Amount)-[:INCOME]->(:Amount)
(:Amount)-[:INCOME]->(:Topic)
(:Person)-[:NEXT]->(:Person)
(:Person)-[:NEXT]->(:Organization)
(:Organization)-[:PART_OF]->(:To

In [ ]:
query = """
MATCH (f:File)
RETURN f
"""
kg.query(query)

[{'f': {'id': 'Http://Www.Un.Org/Sustainabledevelopment',
   'source': 'http://www.un.org/sustainabledevelopment'}},
 {'f': {'id': 'Un_Photo', 'source': '© UN Photo / Hien Macline'}},
 {'f': {'id': 'Un_Photo_Albert_González_Farran',
   'source': '© UN Photo / Albert González Farran'}}]

In [ ]:
query = """
MATCH (c:Chunk)
RETURN count(c)
"""
kg.query(query)

[{'count(c)': 9}]

## 2.3. Semi-Automatic
-> Fixed nodes and relationships, automatic lexical graph, and potential schema

Notes:
- Using native Neo4j functions.
- Embeddings are also generated.

In [ ]:
# Create an Embedder object
embedder = OpenAIEmbeddings(model="text-embedding-3-large")

# Instantiate the LLM
llm = OpenAILLM(
    model_name="gpt-4o",
    model_params={
        "max_tokens": 2000,
        "response_format": {"type": "json_object"},
        "temperature": 0,
    },
)

In [ ]:
# List the entities and relations the LLM should look for in the text
# Note: Used method automatic creates the Lexical Graph
entities = ["Person", "Organization", "Address", 'Amount']
relations = ["LOCATED_AT", "WORKED_AT", "HELPED_BY", "RECEIVED", "SPENT"]
potential_schema = [
    ("Person", "LOCATED_AT", "Address"),
    ("Person", "WORKED_AT", "Organization"),
    ("Person", "HELPED_BY", "Organization"),
    ("Person", "RECEIVED", "Amount"),
    ("Person", "SPENT", "Amount"),
    ("Organization", "LOCATED_AT", "Address"),
    ("Organization", "RECEIVED", "Amount"),
    ("Organization", "SPENT", "Amount"),
]

In [ ]:
# Connect to the Neo4j database
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

# Instantiate the SimpleKGPipeline
kg_builder = SimpleKGPipeline(
    llm=llm,
    driver=driver,
    embedder=embedder,
    entities=entities,
    relations=relations,
    potential_schema=potential_schema,
    perform_entity_resolution=True,
    on_error="IGNORE",
    from_pdf=True,
)

file_names = os.listdir(data_dir)
for index, file_name in enumerate(file_names):
  print(f"[{index+1}/{len(file_names)}] {file_name}")
  file_path = os.path.join(*[data_dir, file_name])
  #asyncio.run(kg_builder.run_async(file_path=file_path))
  await kg_builder.run_async(file_path=file_path)

driver.close()

[1/4] Goal-1.pdf


/usr/lib/python3.10/typing.py:1512: RuntimeWarning: coroutine 'SimpleKGPipeline.run_async' was never awaited
  return super().__instancecheck__(instance)


[2/4] Goal-3.pdf
[3/4] Goal-4.pdf
[4/4] Goal-2.pdf


(See images notebook_1_section_2.3_graph... or explore the databse in Aura)

**WARNING**: It seems like there is a bug in the lexical graph building because the same chunk is connected to several documents.

## 2.4. Semi-Manual
-> Manual Lexical Graph and Semi-automatic domain graph

Notes:
- Manually building the lexical graph by exploring the chunking process
- Using LangChain integration to automatically build the domain graph
- Embeddings are generated manually using native Neo4j functions.

[TBD]